# Experiment 2: linear CKA across LLaVA vision layers

This experiment uses the successfully generated AMP attack-set triplets selected by Experiment 1. Images are observations. For every source, adversarial, and target image, each cached full `[1, 577, 1024]` hidden state is reduced to one 1024-dimensional feature vector by taking the arithmetic mean over **all** tokens, including CLS. All returned hidden states are retained, including the embedding state.

The CKA calculation preserves Google's official [`representation_similarity/Demo.ipynb`](https://github.com/google-research/google-research/blob/master/representation_similarity/Demo.ipynb) formulation: form linear Gram matrices, double-center them, and normalize their Hilbert–Schmidt inner product. CKA is computed independently over the common AMP sample observations for source, adversarial, and target representations; the adversarial-minus-source matrix is a descriptive difference only.


In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from transformers import LlavaForConditionalGeneration

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
DTYPE = torch.float16
DEVICE = torch.device("cuda")
CACHE_VERSION = 1
EXPECTED_STATE_COUNT = 25
EXPECTED_STATE_SHAPE = (1, 577, 1024)
SUCCESS_STATUSES = {"completed", "skipped", "success", "successful"}


def find_repo_root():
    """Find the checkout from either the repository or notebook directory."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / ".git").exists() and (candidate / "generate_amp_perturbations.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the AMP repository root from the current working directory.")


REPO_ROOT = find_repo_root()
ATTACK_DIR = REPO_ROOT / "dataset/laion_art/attack_set"
MANIFEST_PATH = ATTACK_DIR / "manifest.csv"
ATTACK_RESULTS_PATH = ATTACK_DIR / "attack_results.csv"
CACHE_DIR = ATTACK_DIR / "representations/llava_1_5_7b"
OUTPUT_DIR = REPO_ROOT / "adversarial_mislabeling_attack/llava"
OUTPUT_PATHS = {
    "source": OUTPUT_DIR / "exp2_linear_cka_source.csv",
    "adversarial": OUTPUT_DIR / "exp2_linear_cka_adversarial.csv",
    "target": OUTPUT_DIR / "exp2_linear_cka_target.csv",
    "delta": OUTPUT_DIR / "exp2_linear_cka_adv_minus_source.csv",
    "samples": OUTPUT_DIR / "exp2_samples.csv",
}
LAYER_LABELS = ["Emb.", *map(str, range(1, EXPECTED_STATE_COUNT))]


In [ ]:
cache_counts = {"hits": 0, "extracted": 0}
vision_tower = None
to_tensor = transforms.ToTensor()
preprocess = transforms.Compose(
    [
        transforms.Resize((336, 336), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.Normalize(
            (0.48145466, 0.4578275, 0.40821073),
            (0.26862954, 0.26130258, 0.27577711),
        ),
    ]
)


def cache_name(identifier):
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", str(identifier)).strip("._")
    if not value:
        raise ValueError(f"Invalid empty cache identifier: {identifier!r}")
    return f"{value}.pt"


def validate_states(states):
    return (
        isinstance(states, (list, tuple))
        and len(states) == EXPECTED_STATE_COUNT
        and all(
            isinstance(state, torch.Tensor)
            and tuple(state.shape) == EXPECTED_STATE_SHAPE
            and state.dtype == DTYPE
            and state.device.type == "cpu"
            for state in states
        )
    )


def load_cached_states(cache_path):
    try:
        payload = torch.load(cache_path, map_location="cpu", weights_only=True)
        states = payload["hidden_states"]
        metadata = payload["metadata"]
        shapes = [tuple(state.shape) for state in states]
        compatible = (
            metadata.get("cache_version") == CACHE_VERSION
            and metadata.get("model_id") == MODEL_ID
            and metadata.get("hidden_state_count") == len(states)
            and [tuple(shape) for shape in metadata.get("tensor_shapes", [])] == shapes
            and validate_states(states)
        )
        if not compatible:
            raise ValueError("incompatible cache metadata or tensors")
        cache_counts["hits"] += 1
        return tuple(states)
    except Exception as error:
        warnings.warn(f"Ignoring invalid representation cache {cache_path}: {error}")
        return None


def get_vision_tower():
    """Load LLaVA lazily, so a cache-complete run never initializes the model."""
    global vision_tower
    if vision_tower is None:
        if not torch.cuda.is_available():
            raise RuntimeError("A CUDA GPU is required to recompute missing LLaVA representations.")
        llava_model = LlavaForConditionalGeneration.from_pretrained(
            MODEL_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True
        )
        vision_tower = llava_model.vision_tower.to(DEVICE).eval()
        config = vision_tower.config
        actual_shape = (
            1,
            (config.image_size // config.patch_size) ** 2 + 1,
            config.hidden_size,
        )
        if config.num_hidden_layers + 1 != EXPECTED_STATE_COUNT or actual_shape != EXPECTED_STATE_SHAPE:
            raise ValueError(
                f"Incompatible LLaVA vision tower: states={config.num_hidden_layers + 1}, "
                f"shape={actual_shape}"
            )
        del llava_model
    return vision_tower


def extract_hidden_states(image_path):
    """Return every full FP16 vision hidden-state token tensor on CPU."""
    model = get_vision_tower()
    with Image.open(image_path) as image:
        image_tensor = to_tensor(image.convert("RGB")).to(DEVICE, DTYPE)
    pixel_values = preprocess(image_tensor).unsqueeze(0)
    with torch.inference_mode():
        outputs = model(pixel_values, output_hidden_states=True)
    states = tuple(state.detach().to(device="cpu", dtype=DTYPE) for state in outputs.hidden_states)
    if not validate_states(states):
        raise ValueError(
            f"Unexpected hidden states for {image_path}: "
            f"count={len(states)}, shapes={[tuple(state.shape) for state in states]}"
        )
    return states


def get_hidden_states(image_path, cache_path):
    if cache_path.is_file():
        cached = load_cached_states(cache_path)
        if cached is not None:
            return cached
    states = extract_hidden_states(image_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "metadata": {
                "cache_version": CACHE_VERSION,
                "model_id": MODEL_ID,
                "hidden_state_count": len(states),
                "tensor_shapes": [tuple(state.shape) for state in states],
                "dtype": str(DTYPE),
            },
            "hidden_states": states,
        },
        cache_path,
    )
    cache_counts["extracted"] += 1
    return states


def validate_image(path):
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as image:
        image.verify()


def mean_pool(states):
    """Produce [layers, 1024] float32 features by averaging every token, including CLS."""
    return torch.stack([state.float().mean(dim=1).squeeze(0) for state in states]).numpy()


In [ ]:
# Join in manifest order with the same successful-status rule as Experiment 1.
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
attack_results = pd.read_csv(ATTACK_RESULTS_PATH, dtype=str)
required_manifest = {
    "sample_id", "pair_id", "source_path", "target_path", "source_image_id",
    "target_image_id", "source_concept", "target_concept",
}
required_results = {"sample_id", "adv_path", "status"}
for path, frame, required in [
    (MANIFEST_PATH, manifest, required_manifest),
    (ATTACK_RESULTS_PATH, attack_results, required_results),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {sorted(missing)}")
    if frame["sample_id"].duplicated().any():
        raise ValueError(f"Duplicate sample IDs in {path}")

samples = manifest.merge(
    attack_results[["sample_id", "adv_path", "status"]],
    on="sample_id",
    how="left",
    sort=False,
    validate="one_to_one",
)
provenance_columns = [
    "sample_id", "pair_id", "source_image_id", "target_image_id",
    "source_concept", "target_concept",
]
feature_rows = {"source": [], "adversarial": [], "target": []}
provenance_rows = []
skipped = []

for sample in samples.itertuples(index=False):
    status = str(sample.status).strip().lower()
    if status not in SUCCESS_STATUSES:
        reason = f"attack status={sample.status!r}"
        skipped.append((sample.sample_id, reason))
        warnings.warn(f"Skipping {sample.sample_id}: {reason}")
        continue

    paths = {
        "source": REPO_ROOT / sample.source_path,
        "adversarial": REPO_ROOT / sample.adv_path,
        "target": REPO_ROOT / sample.target_path,
    }
    cache_paths = {
        "source": CACHE_DIR / "clean" / cache_name(sample.source_image_id),
        "adversarial": CACHE_DIR / "adv" / cache_name(sample.sample_id),
        "target": CACHE_DIR / "clean" / cache_name(sample.target_image_id),
    }
    try:
        # Reject missing/corrupt members before adding any part of the triplet.
        for image_path in paths.values():
            validate_image(image_path)
        pooled = {
            condition: mean_pool(get_hidden_states(paths[condition], cache_paths[condition]))
            for condition in ("source", "adversarial", "target")
        }
        if any(value.shape != (EXPECTED_STATE_COUNT, EXPECTED_STATE_SHAPE[-1]) for value in pooled.values()):
            raise ValueError(f"Unexpected pooled representation shapes: { {k: v.shape for k, v in pooled.items()} }")
        for condition in feature_rows:
            feature_rows[condition].append(pooled[condition])
        provenance_rows.append({column: getattr(sample, column) for column in provenance_columns})
    except Exception as error:
        reason = f"{type(error).__name__}: {error}"
        skipped.append((sample.sample_id, reason))
        warnings.warn(f"Skipping {sample.sample_id}: {reason}")

valid_count = len(provenance_rows)
if valid_count < 3:
    raise RuntimeError(
        f"Linear CKA requires at least 3 valid AMP samples; only {valid_count} passed "
        "status, image, and representation validation."
    )

# Each stack begins [samples, layers, width]; transpose to [layers, samples, width].
source_features = np.stack(feature_rows["source"], axis=0).transpose(1, 0, 2)
adversarial_features = np.stack(feature_rows["adversarial"], axis=0).transpose(1, 0, 2)
target_features = np.stack(feature_rows["target"], axis=0).transpose(1, 0, 2)
print("feature arrays [layers, samples, dimensions]:", {
    "source": source_features.shape,
    "adversarial": adversarial_features.shape,
    "target": target_features.shape,
})


In [ ]:
# Adapted directly from Google's public CKA demo formulation.
def center_gram(gram, unbiased=False):
    if not np.allclose(gram, gram.T):
        raise ValueError("Input must be a symmetric Gram matrix.")
    gram = gram.copy()
    if unbiased:
        np.fill_diagonal(gram, 0)
        means = np.sum(gram, axis=0, dtype=np.float64) / (gram.shape[0] - 2)
        means -= np.sum(means) / (2 * (gram.shape[0] - 1))
        gram -= means[:, None]
        gram -= means[None, :]
        np.fill_diagonal(gram, 0)
    else:
        means = np.mean(gram, axis=0, dtype=np.float64)
        means -= np.mean(means) / 2
        gram -= means[:, None]
        gram -= means[None, :]
    return gram


def cka(gram_x, gram_y, debiased=False):
    gram_x = center_gram(gram_x, unbiased=debiased)
    gram_y = center_gram(gram_y, unbiased=debiased)
    scaled_hsic = gram_x.ravel().dot(gram_y.ravel())
    normalization_x = np.linalg.norm(gram_x)
    normalization_y = np.linalg.norm(gram_y)
    return scaled_hsic / (normalization_x * normalization_y)


def linear_cka_matrix(layer_features, debiased=False):
    """Compute all layer pairs with images as Gram-matrix observations."""
    grams = [layer @ layer.T for layer in layer_features]
    count = len(grams)
    matrix = np.empty((count, count), dtype=np.float64)
    for row in range(count):
        for column in range(row, count):
            value = cka(grams[row], grams[column], debiased=debiased)
            matrix[row, column] = matrix[column, row] = value
    return matrix


cka_source = linear_cka_matrix(source_features, debiased=False)
cka_adversarial = linear_cka_matrix(adversarial_features, debiased=False)
cka_target = linear_cka_matrix(target_features, debiased=False)
cka_delta = cka_adversarial - cka_source

frames = {
    "source": pd.DataFrame(cka_source, index=LAYER_LABELS, columns=LAYER_LABELS),
    "adversarial": pd.DataFrame(cka_adversarial, index=LAYER_LABELS, columns=LAYER_LABELS),
    "target": pd.DataFrame(cka_target, index=LAYER_LABELS, columns=LAYER_LABELS),
    "delta": pd.DataFrame(cka_delta, index=LAYER_LABELS, columns=LAYER_LABELS),
}
for condition, frame in frames.items():
    frame.to_csv(OUTPUT_PATHS[condition], index_label="layer")  # Replace, never append.
pd.DataFrame(provenance_rows, columns=provenance_columns).to_csv(
    OUTPUT_PATHS["samples"], index=False
)
frames["adversarial"]


In [ ]:
def draw_cka_heatmap(axis, matrix, title, cmap, vmin, vmax, colorbar_label):
    image = axis.imshow(matrix, vmin=vmin, vmax=vmax, cmap=cmap, origin="upper")
    axis.set_xticks(range(len(LAYER_LABELS)), labels=LAYER_LABELS, rotation=90)
    axis.set_yticks(range(len(LAYER_LABELS)), labels=LAYER_LABELS)
    axis.set_xlabel("Layer")
    axis.set_ylabel("Layer")
    axis.set_title(title)
    axis.figure.colorbar(image, ax=axis, label=colorbar_label)


difference_bound = float(np.max(np.abs(cka_delta)))
if difference_bound == 0:
    difference_bound = np.finfo(np.float64).eps
fig, axes = plt.subplots(2, 2, figsize=(18, 15))
draw_cka_heatmap(
    axes[0, 0], cka_source,
    f"LLaVA-1.5-7B source linear CKA (N={valid_count} AMP samples)",
    "viridis", 0, 1, "Linear CKA",
)
draw_cka_heatmap(
    axes[0, 1], cka_adversarial,
    f"LLaVA-1.5-7B adversarial linear CKA (N={valid_count} AMP samples)",
    "viridis", 0, 1, "Linear CKA",
)
draw_cka_heatmap(
    axes[1, 0], cka_target,
    f"LLaVA-1.5-7B target linear CKA (N={valid_count} AMP samples)",
    "viridis", 0, 1, "Linear CKA",
)
draw_cka_heatmap(
    axes[1, 1], cka_delta,
    f"LLaVA-1.5-7B adversarial − source CKA (N={valid_count} AMP samples)",
    "coolwarm", -difference_bound, difference_bound, "Linear CKA difference",
)
fig.tight_layout()
plt.show()

print(
    "Summary: "
    f"manifest={len(manifest)}, valid/analyzed={valid_count}, skipped={len(skipped)}, "
    f"cache_hits={cache_counts['hits']}, newly_extracted={cache_counts['extracted']}"
)
if skipped:
    print("Skipped samples:", "; ".join(f"{sample_id} ({reason})" for sample_id, reason in skipped))
